In [ ]:
import torch
print("Torch version:", torch.__version__)              #Basicamente en un entorno virtual echo con anaconda, 
print("CUDA disponible:", torch.cuda.is_available())    #se usa python 3.11.15, y se necesita tener pytorch instalado con sus depedencias de CUDA correspondientes
print("CUDA version compilada:", torch.version.cuda)    #prints comentados por si se quiere revisar resultados de algun bloque de codigo

if torch.cuda.is_available():
    print("GPU disponible:", torch.cuda.get_device_name(0))
else: 
    print("Algo ha ido mal. No se ha detectado GPU compatible con CUDA")
    
#S1: "My voice is my Password"
#S2: "Okay Miguel"
#S3: "Artificial intelligence is for real"
#S4: "Actions speak louder than words"
#S5: "There is no such thing as a free lunch"


   
import librosa                      #para cargar el audio
import os                           #para moverme entre directorios usando el S.O.
import matplotlib.pyplot as plt     #para visualizar el audio
import numpy                        #para manejar los datos de audio como arrays

In [ ]:
#Aquí recorremos la base de datos que nos pasó el profesor para "guardar los paths"

dirRaiz = "Database"   
data = []
locutoresMap = {}
locutoresCont = 0

fraseMap = {}
fraseCont = 0

for frase in os.listdir(dirRaiz):
    if frase not in fraseMap:
        fraseMap[frase] = fraseCont
        fraseCont += 1
#print(fraseMap)

for frase in os.listdir(dirRaiz):
    frasePath = os.path.join(dirRaiz, frase)

    for locutor in os.listdir(frasePath):
        if locutor not in locutoresMap:
            locutoresMap[locutor] = locutoresCont
            locutoresCont += 1

        locutorEtiqueta = locutoresMap[locutor]
        locutorPath = os.path.join(frasePath, locutor)

        for file in os.listdir(locutorPath):
            archivoPath = os.path.join(locutorPath, file)

            data.append({
                "path": archivoPath,
                "locutor": locutorEtiqueta,
                "frase": fraseMap[frase]
            })

print(len(data))
print(data[8])

In [ ]:
from torch.utils.data import Dataset

class LocutoresDataset(Dataset):                    #creare la clase del dataset para poder hacer de forma facil los tensores para pytorch

    def __init__(self, data):  #Constructor recibira data[]
        self.data = data

    def __len__(self):          #Para ver la longitud
        return len(self.data)

    def __getitem__(self, indice):     
        item = self.data[indice]                #carga los datos de data[]
        archivoPath = item["path"]              #carga el path del audio        
        locutorEtiqueta = item["locutor"]       #carga el locutor del audio

        waveform, sr = librosa.load(archivoPath,sr=16000)                 #carga audio de archivoPath
        mfcc = librosa.feature.mfcc(y = waveform, sr = sr, n_mfcc=40)     #extrae los MFCC del audio cargado

        tensorMFCC = torch.tensor(mfcc, dtype=torch.float32)               #convierte los MFCC a tensor de PyTorch
        tensorMFCC = tensorMFCC.unsqueeze(0)                               #agrega una dimensión adicional al tensor para representar el canal (en este caso, 1 canal para MFCC)

        tensorLocutor = torch.tensor(locutorEtiqueta)               #convierte la etiqueta del locutor a tensor de PyTorch
        return tensorMFCC, tensorLocutor

In [ ]:
#Creo el dataset
dataset = LocutoresDataset(data)

#Lo pruebo
mfcc, locutor = dataset[90]

print(mfcc.shape)
print(locutor)

plt.imshow(mfcc[0].numpy())
plt.show()